# Multi-Scale Attention-Augmented Network with Strict Ordinal Loss (CORAL) for Knee OA Grading
### Applied Digital Image Processing (DIP) Pipeline
**Inspired by:**
- *Zhang et al. (Frontiers in Medicine, June 2026)*: Multi-scale attention-guided framework (FPN + SE Channel Attention + Spatial Self-Attention).
- *Cao et al. (CORAL)*: Consistent Rank Logits for strict ordinal disease progression.
- *Classic DIP Principles*: Butterworth 2D FFT filtering, CLAHE, GLCM Sclerosis analysis, and LBP osteophyte descriptors.

*Runs seamlessly on Local CPU or Google Colab (CPU / T4 GPU).*


In [ ]:
# 1. Environment Setup & Hardware Check
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

# Add Project root to Python path
PROJECT_ROOT = os.path.abspath('..') if os.path.exists('../utils') else os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {tf.keras.__version__}")

# GPU Configuration & Memory Growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[GPU ACTIVE] Found {len(gpus)} GPU(s): {gpus}")
        # Enable Mixed Precision FP16 on GPU (cuts VRAM usage in half)
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("[MIXED PRECISION] Enabled 'mixed_float16' for accelerated training.")
    except Exception as e:
        print(f"GPU Config note: {e}")
else:
    print("[CPU MODE] Running on CPU. EfficientNet-B0 backbone recommended for fast iteration.")


In [ ]:
# 2. Import Modular Upgrades
from utils.ordinal_loss import (
    grade_to_ordinal_vector,
    batch_grades_to_ordinal,
    ordinal_probs_to_grade,
    ordinal_probs_to_class_distribution,
    CoralOrdinalLayer,
    CoralLoss,
    EarthMoversDistanceLoss,
    compute_quadratic_weighted_kappa,
    compute_ordinal_mae
)
from utils.attention_backbone import build_multiscale_attention_model
from utils.dip_processing import (
    apply_clahe,
    extract_knee_joint_roi,
    butterworth_lowpass_filter,
    homomorphic_filter,
    extract_glcm_features,
    extract_lbp_features,
    preprocess_radiograph_full_dip
)

print("All modular DIP, Attention, and Ordinal packages imported successfully!")


## 3. Dataset Discovery & Class Distribution Analysis
The dataset contains digital knee radiographs graded on the Kellgren-Lawrence (KL) scale (0: Normal to 4: Severe).


In [ ]:
# Locate Dataset Directory
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset-kaggle")
train_dir = os.path.join(DATASET_DIR, "train")
val_dir = os.path.join(DATASET_DIR, "val")
test_dir = os.path.join(DATASET_DIR, "test")

def get_image_records(folder):
    records = []
    for grade in range(5):
        grade_folder = os.path.join(folder, str(grade))
        for ext in ("*.png", "*.jpg", "*.jpeg"):
            for f in glob.glob(os.path.join(grade_folder, ext)):
                records.append({"path": f, "grade": grade})
    return pd.DataFrame(records)

df_train = get_image_records(train_dir)
df_val = get_image_records(val_dir)
df_test = get_image_records(test_dir)

print(f"Train Samples: {len(df_train)}")
print(f"Val Samples:   {len(df_val)}")
print(f"Test Samples:  {len(df_test)}")

# Plot Class Distribution
plt.figure(figsize=(8, 3.5))
ax = df_train["grade"].value_counts().sort_index().plot(kind="bar", color="#1f77b4", edgecolor="black")
plt.title("Training Set KL Grade Distribution (Notice 13:1 Class Imbalance)")
plt.xlabel("KL Severity Grade (0: Normal -> 4: Severe)")
plt.ylabel("Radiograph Count")
plt.xticks(rotation=0)
for p in ax.patches:
    ax.annotate(str(p.get_height()), (p.get_x() + 0.08, p.get_height() + 30))
plt.tight_layout()
plt.show()

# Compute Balanced Class Weights
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(df_train["grade"].values)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=df_train["grade"].values)
class_weights_dict = dict(zip(classes, weights))
print("Balanced Class Weights:", class_weights_dict)


## 4. Digital Image Processing (DIP) Visual Demonstration
Before training, let's visualize how the classical mathematical DIP operators (Sobel ROI, Butterworth FFT, and CLAHE) clean the raw X-ray data.


In [ ]:
# Visualizing DIP Preprocessing Pipeline on a Sample Knee X-Ray
sample_row = df_train[df_train["grade"] == 3].iloc[0]
raw_sample = cv2.imread(sample_row["path"], cv2.IMREAD_GRAYSCALE)

# 1. CLAHE
clahe_img = apply_clahe(raw_sample)

# 2. Butterworth 2D FFT Low-pass
fft_img = butterworth_lowpass_filter(raw_sample, cutoff_freq=45, order=2)

# 3. Homomorphic Filter
homo_img = homomorphic_filter(raw_sample)

# 4. End-to-End Combined Preprocessed
final_dip = preprocess_radiograph_full_dip(raw_sample, target_size=(224, 224))

plt.figure(figsize=(15, 3.5))
plt.subplot(1, 5, 1); plt.imshow(raw_sample, cmap="gray"); plt.title("1. Raw X-Ray"); plt.axis("off")
plt.subplot(1, 5, 2); plt.imshow(clahe_img, cmap="gray"); plt.title("2. Spatial CLAHE"); plt.axis("off")
plt.subplot(1, 5, 3); plt.imshow(fft_img, cmap="gray"); plt.title("3. 2D FFT Butterworth"); plt.axis("off")
plt.subplot(1, 5, 4); plt.imshow(homo_img, cmap="gray"); plt.title("4. Homomorphic Filter"); plt.axis("off")
plt.subplot(1, 5, 5); plt.imshow(final_dip); plt.title("5. Model Ready (DIP Fused)"); plt.axis("off")
plt.tight_layout()
plt.show()

# Extract and display GLCM & LBP texture features
glcm = extract_glcm_features(raw_sample)
print("GLCM Sclerosis Features (Bone Density Measure):", glcm)


## 5. Multi-Scale Attention Model Construction (Zhang et al. 2026)
We build the **Feature Pyramid Network (FPN) + Squeeze-and-Excitation (SE) Channel Attention + Spatial Self-Attention** model terminating in our **CORAL Ordinal Layer**.


In [ ]:
# Build Model: EfficientNet-B2 with STN Joint Localisation & CORAL Head
BACKBONE_CHOICE = "b2"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32 if gpus else 16

print(f"Selected Backbone: EfficientNet-{BACKBONE_CHOICE.upper()} with Spatial Transformer Network (STN)")
model = build_multiscale_attention_model(
    backbone_type=BACKBONE_CHOICE,
    input_shape=(*IMG_SIZE, 3),
    pyramid_channels=128,
    num_classes=5,
    head_type="coral",
    dropout_rate=0.3,
    pretrained=True,
    use_stn=True
)

print(f"Total Model Parameters: {model.count_params():,}")
print("Model Inputs:", model.input_shape)
print("Model Outputs:", model.output_shape)


## 6. High-Performance tf.data Input Pipeline
Applies on-the-fly DIP preprocessing (Butterworth + CLAHE), data augmentation (rotations, horizontal flips for laterality), and CORAL multi-threshold target encoding.


In [ ]:
def tf_dip_preprocess(img_path_tensor, grade_tensor):
    def _py_loader(path_bytes, grade_val):
        path = path_bytes.decode('utf-8')
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((*IMG_SIZE,), dtype=np.uint8)
        # Apply Butterworth 2D FFT + CLAHE
        processed = preprocess_radiograph_full_dip(img, target_size=IMG_SIZE)
        # Convert integer grade to 4-element CORAL binary vector
        ordinal_vec = grade_to_ordinal_vector(int(grade_val), num_classes=5)
        return processed.astype(np.float32), ordinal_vec.astype(np.float32)

    img, target = tf.py_function(
        _py_loader,
        [img_path_tensor, grade_tensor],
        [tf.float32, tf.float32]
    )
    img.set_shape([*IMG_SIZE, 3])
    target.set_shape([4])
    return img, target

# Augmentation layer
augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomZoom(0.06),
])

def create_dataset(df, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["grade"].values))
    if is_training:
        ds = ds.shuffle(buffer_size=1024, seed=42)
    ds = ds.map(tf_dip_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_dataset = create_dataset(df_train, is_training=True)
val_dataset = create_dataset(df_val, is_training=False)
test_dataset = create_dataset(df_test, is_training=False)

print("tf.data pipelines ready for streaming execution!")


## 7. Two-Phase Training Strategy (Warmup + Fine-Tuning)
- **Phase 1 (Warmup)**: Freeze backbone spine; train only FPN + Attention + CORAL head.
- **Phase 2 (Deep Fine-Tuning)**: Unfreeze backbone with a low learning rate ($10^{-5}$) to adapt to medical radiographs.


In [ ]:
# Instantiate Strict CORAL Loss with Class Imbalance Penalties
class_weight_list = [float(class_weights_dict[i]) for i in range(5)]
coral_loss_fn = CoralLoss(class_weights=class_weight_list)

# PHASE 1: Freeze base convolutional backbone
base_prefix = "efficientnet"
for layer in model.layers:
    if base_prefix in layer.name.lower():
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=coral_loss_fn,
    metrics=["binary_accuracy"]
)

callbacks_p1 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_multiscale_coral_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

print("=== Starting Phase 1: Head and Attention Warmup ===")
history_p1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=6,
    callbacks=callbacks_p1
)


In [ ]:
# PHASE 2: Unfreeze Backbone for Fine-Tuning
print("=== Starting Phase 2: End-to-End Fine-Tuning ===")
for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=coral_loss_fn,
    metrics=["binary_accuracy"]
)

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_multiscale_coral_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

history_p2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=12,
    callbacks=callbacks_p2
)


## 8. Evaluation: Medical Ordinal Metrics (QWK, MAE, Classification Report)
We evaluate the fine-tuned model on unseen test radiographs, testing Quadratic Weighted Kappa (penalizing non-adjacent clinical errors) and per-grade F1 scores.


In [ ]:
# Predict on Held-Out Test Set
print("Running inference on held-out test dataset...")
y_pred_probs = model.predict(test_dataset)

# Convert 4 cumulative probabilities to discrete KL grade (0..4)
y_pred_grades = ordinal_probs_to_grade(y_pred_probs, threshold=0.5)
y_true_grades = df_test["grade"].values[:len(y_pred_grades)]

# Calculate Clinical Metrics
qwk = compute_quadratic_weighted_kappa(y_true_grades, y_pred_grades)
mae = compute_ordinal_mae(y_true_grades, y_pred_grades)

print(f"\n" + "="*40)
print(f"Quadratic Weighted Kappa (QWK): {qwk:.4f}")
print(f"Mean Absolute Error (MAE):       {mae:.4f}")
print("="*40 + "\n")

# Classification Report
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

target_names = [f"Grade {i}" for i in range(5)]
print(classification_report(y_true_grades, y_pred_grades, target_names=target_names, digits=4))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_true_grades, y_pred_grades)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix — Multi-Scale Attention + CORAL")
plt.xlabel("Predicted Grade")
plt.ylabel("True Grade")
plt.tight_layout()
plt.show()


## 9. Grad-CAM Spatial Explainability
Visualizing the attention activations on osteophytes and joint space narrowing across the multi-scale attention heads.


In [ ]:
# Grad-CAM on Attention Fused Output
def generate_multiscale_gradcam(img_array, model):
    target_layer = model.get_layer("fuse_mid")
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[target_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_output, preds = grad_model(img_array)
        # Sum predicted probabilities across ordinal thresholds
        score = tf.reduce_sum(preds[0])

    grads = tape.gradient(score, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Test Grad-CAM on a test radiograph
sample_test_path = df_test[df_test["grade"] >= 3].iloc[0]["path"]
raw_test_img = cv2.imread(sample_test_path, cv2.IMREAD_GRAYSCALE)
test_prep = preprocess_radiograph_full_dip(raw_test_img, target_size=IMG_SIZE)
test_input = np.expand_dims(test_prep, axis=0)

cam_heatmap = generate_multiscale_gradcam(test_input, model)
cam_resized = cv2.resize(cam_heatmap, (raw_test_img.shape[1], raw_test_img.shape[0]))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(cv2.cvtColor(raw_test_img, cv2.COLOR_GRAY2BGR), 0.6, heatmap_colored, 0.4, 0)

plt.figure(figsize=(10, 4.5))
plt.subplot(1, 2, 1); plt.imshow(raw_test_img, cmap="gray"); plt.title("Original Radiograph"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); plt.title("Multi-Scale Attention Grad-CAM"); plt.axis("off")
plt.tight_layout()
plt.show()
